# JSON

Clickhouse supports the JSON data type. Although it has many functions for processing strings that represent JSON, but this section focuses on the operating exactly with JSON datatype.

## Reading paths

The most common task is to loading values stored under sertaing columns in JSON.

To extract values by JSON path refer to path with subcolumns: `<column name>.<attribute 1>.<attribute 2>.<...>`. If the type is not specified in the column declaration, the output **always** will be of type `Dynamic`.

**Note** there are [`JSONExtract` functions](https://clickhouse.com/docs/sql-reference/functions/json-functions#jsonextract-functions) but they are designed to extract values from string formatted as JSON.

**Note** refering to the paths that are JSONs by themselves resutls in `NULL` wrapped in `Dynamic`.

---

The following cell shows how the paths are parsed from the JSON attributes:

In [27]:
--ClickHouse
SELECT
    json,
    json.a,
    JSONExtract(json::String, 'a', 'UInt8')
FROM values(
    'json JSON',
    ('{"a": 10}'),
    ('{"a": 20, "b": "string"}'),
    ('{"a": 40}')
);

json,json.a,"JSONExtract(CAST(json, 'String'), 'a', 'UInt8')"
{'a': 10},10,10
"{'a': 20, 'b': 'string'}",20,20
{'a': 40},40,40


The following example shows the parsing in the case of paths with complex structure.

In [44]:
--ClickHouse
SELECT
    '{"a": [10, 20, 30], "b": {"c": "value"}}'::JSON as inp_col,
    inp_col.a,
    toTypeName(inp_col.a),
    dynamicType(inp_col.a),
    inp_col.b,
    toTypeName(inp_col.b),
    dynamicType(inp_col.b);

inp_col,inp_col.a,toTypeName(inp_col.a),dynamicType(inp_col.a),inp_col.b,toTypeName(inp_col.b),dynamicType(inp_col.b)
"{'a': [10, 20, 30], 'b': {'c': 'value'}}","[10, 20, 30]",Dynamic,Array(Nullable(Int64)),,Dynamic,None


Note that `inp_col.b` is not parsed correctly.

## Arrays

Accessing to the array under some key of JSON returns a dynamic data type that can be parsed as an `Array`. By explicitly casing the subcolumns syntax to `Array(T)`, you can operate on this opject as you would on regular array. Specifically `Array(JSON)`.

There is an alias for JSON paths: `[]`. Using this syntax after a JSON column is equivalent to explicitly casting to JSON.

The great thing about `Array(JSON)` is that you can access the particular attribute of each JSON-object in the array using the subcolumns syntax.

Check the [Handling array of JSON objects](https://clickhouse.com/docs/sql-reference/data-types/newjson#handling-arrays-of-json-objects) section.

---

The following code snippet shows the types returned when accessing a JSON field containing an array:

In [53]:
--ClickHouse
SELECT
    '{"a": [10, 20, 30]}'::JSON as inp_col,
    toTypeName(inp_col.a),
    dynamicType(inp_col.a);

inp_col,toTypeName(inp_col.a),dynamicType(inp_col.a)
"{'a': [10, 20, 30]}",Dynamic,Array(Nullable(Int64))


The explicit casting to `JSON` allows the received object to be operated as if it were a regular `Array(T)`:

In [72]:
--ClickHouse
SELECT
    '{"a": [{"b": 20, "c": 40, "m": 40}]}'::JSON as inp_col,
    inp_col.a::Array(JSON) as array,
    toTypeName(array);

inp_col,array,toTypeName(array),array.b
"{'a': [{'b': 20, 'c': 40, 'm': 40}]}","[{'b': 20, 'c': 40, 'm': 40}]",Array(JSON),[20]


In practice, the syntax `[]` allows to conveniently specify paths hidden under arrays when working with JSONs with lots of nested attributes:

In [73]:
--ClickHouse
SELECT
    '{"a": [{"b": 20, "c": 40}, {"b": 50, "c": 10}]}'::JSON as inp_col,
    inp_col.a[].b as b;

inp_col,b
"{'a': [{'b': 20, 'c': 40}, {'b': 50, 'c': 10}]}","[20, 50]"
